# Cognitive Stress Model

Visualizing physiological signals and self-reported stress levels from 22 subjects.

**Subjects:**
- **V1 (males with Stroop):** S04, S05, S08, S09, S10, S13, S14, S15, S17, S18
- **V2 (females without Stroop):** f01, f02, f03, f04, f05, f06, f08, f09, f10, f11, f12, f13

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import datetime

plt.style.use('seaborn-v0_8-whitegrid')

In [ ]:
# Paths
dataset_path = '22subjects/STRESS'
stress_level_v1_path = 'WISE_data_files/Stress_Level_v1.csv'
stress_level_v2_path = 'WISE_data_files/Stress_Level_v2.csv'

# Subject lists
v1_subjects = ['S04', 'S05', 'S08', 'S09', 'S10', 'S13', 'S14', 'S15', 'S17', 'S18']
v2_subjects = ['f01', 'f02', 'f03', 'f04', 'f05', 'f06', 'f08', 'f09', 'f10', 'f11', 'f12', 'f13']
all_subjects = v1_subjects + v2_subjects

print(f"V1 Subjects (n={len(v1_subjects)}): {v1_subjects}")
print(f"V2 Subjects (n={len(v2_subjects)}): {v2_subjects}")
print(f"Total: {len(all_subjects)} subjects")

---
## Self-Reported Stress Levels

In [ ]:
# Load V1 stress levels
stress_level_v1 = pd.read_csv(stress_level_v1_path, index_col=0)

print("V1 Stress Levels (Males with Stroop Test)")
print(f"Phases: {list(stress_level_v1.columns)}")
print(f"Subjects: {list(stress_level_v1.index)}")
stress_level_v1

In [ ]:
# Load V2 stress levels
stress_level_v2 = pd.read_csv(stress_level_v2_path, index_col=0)

print("V2 Stress Levels (Females without Stroop Test)")
print(f"Phases: {list(stress_level_v2.columns)}")
print(f"Subjects: {list(stress_level_v2.index)}")
stress_level_v2

In [ ]:
# Plot stress levels across phases
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# V1
for subject, row in stress_level_v1.iterrows():
    axes[0].plot(row.index, row.values, marker='o', alpha=0.7, label=subject)
axes[0].set_title('V1: Self-Reported Stress (Males with Stroop)', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Protocol Phase')
axes[0].set_ylabel('Stress Level (0-10)')
axes[0].legend(bbox_to_anchor=(1.02, 1), fontsize=8)
axes[0].grid(True, alpha=0.3)
axes[0].set_ylim(0, 10)
axes[0].tick_params(axis='x', rotation=45)

# V2
for subject, row in stress_level_v2.iterrows():
    axes[1].plot(row.index, row.values, marker='o', alpha=0.7, label=subject)
axes[1].set_title('V2: Self-Reported Stress (Females without Stroop)', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Protocol Phase')
axes[1].set_ylabel('Stress Level (0-10)')
axes[1].legend(bbox_to_anchor=(1.02, 1), fontsize=8)
axes[1].grid(True, alpha=0.3)
axes[1].set_ylim(0, 10)
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
# Mean stress by phase
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# V1
mean_v1 = stress_level_v1.mean()
mean_v1.plot(kind='bar', ax=axes[0], color='steelblue', edgecolor='black', alpha=0.8)
axes[0].set_title('V1: Mean Stress by Phase', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Mean Stress Level')
axes[0].set_xlabel('Phase')
axes[0].axhline(y=5, color='red', linestyle='--', alpha=0.5, label='Moderate (5)')
axes[0].set_ylim(0, 8)
axes[0].tick_params(axis='x', rotation=45)
axes[0].legend()
axes[0].grid(True, axis='y', alpha=0.3)

# V2
mean_v2 = stress_level_v2.mean()
mean_v2.plot(kind='bar', ax=axes[1], color='coral', edgecolor='black', alpha=0.8)
axes[1].set_title('V2: Mean Stress by Phase', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Mean Stress Level')
axes[1].set_xlabel('Phase')
axes[1].axhline(y=5, color='red', linestyle='--', alpha=0.5, label='Moderate (5)')
axes[1].set_ylim(0, 8)
axes[1].tick_params(axis='x', rotation=45)
axes[1].legend()
axes[1].grid(True, axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

# Print rankings
print("V1 - Phases ranked by stress:")
for phase, val in mean_v1.sort_values(ascending=False).items():
    print(f"  {phase}: {val:.2f}")

print("\nV2 - Phases ranked by stress:")
for phase, val in mean_v2.sort_values(ascending=False).items():
    print(f"  {phase}: {val:.2f}")

---
## Load Physiological Signals

In [ ]:
def create_df_array(dataframe):
    return dataframe.values.flatten()

def time_abs_(UTC_array):
    new_array = []
    start_time = datetime.datetime.strptime(UTC_array[0], '%Y-%m-%d %H:%M:%S')
    for utc in UTC_array:
        current_time = datetime.datetime.strptime(utc, '%Y-%m-%d %H:%M:%S')
        seconds_elapsed = (current_time - start_time).total_seconds()
        new_array.append(int(seconds_elapsed))
    return new_array

def moving_average(acc_data):
    avg = 0
    prevX, prevY, prevZ = 0, 0, 0
    results = []
    for i in range(0, len(acc_data), 32):
        sum_ = 0
        buffX = acc_data[i:i+32, 0]
        buffY = acc_data[i:i+32, 1]
        buffZ = acc_data[i:i+32, 2]
        for j in range(len(buffX)):
            sum_ += max(abs(buffX[j] - prevX), abs(buffY[j] - prevY), abs(buffZ[j] - prevZ))
            prevX, prevY, prevZ = buffX[j], buffY[j], buffZ[j]
        avg = avg * 0.9 + (sum_ / 32) * 0.1
        results.append(avg)
    return results

print("Helper functions defined")

In [ ]:
def read_signals(main_folder):
    signal_dict = {}
    time_dict = {}
    fs_dict = {}
    
    subfolders = next(os.walk(main_folder))[1]
    
    utc_start_dict = {}
    for folder_name in subfolders:
        csv_path = f'{main_folder}/{folder_name}/EDA.csv'
        df = pd.read_csv(csv_path)
        utc_start_dict[folder_name] = df.columns.tolist()
    
    for folder_name in subfolders:
        folder_path = os.path.join(main_folder, folder_name)
        files = os.listdir(folder_path)
        
        signals = {}
        time_line = {}
        fs_signal = {}
        
        desired_files = ['EDA.csv', 'BVP.csv', 'HR.csv', 'TEMP.csv', 'tags.csv', 'ACC.csv', 'IBI.csv']
        
        for file_name in files:
            if file_name not in desired_files:
                continue
            
            file_path = os.path.join(folder_path, file_name)
            signal_name = file_name.replace('.csv', '')
            
            if file_name == 'tags.csv':
                try:
                    df = pd.read_csv(file_path, header=None)
                    tags_vector = create_df_array(df)
                    tags_UTC_vector = np.insert(tags_vector, 0, utc_start_dict[folder_name])
                    signal_array = time_abs_(tags_UTC_vector)
                except pd.errors.EmptyDataError:
                    signal_array = []
            elif file_name == 'IBI.csv':
                df = pd.read_csv(file_path)
                signal_array = df.values
                fs_signal['IBI'] = 'variable'
            else:
                df = pd.read_csv(file_path)
                fs = int(df.iloc[0, 0])
                signal_array = df.iloc[1:].values
                time_array = np.linspace(0, len(signal_array)/fs, len(signal_array))
                time_line[signal_name] = time_array
                fs_signal[signal_name] = fs
            
            signals[signal_name] = signal_array
        
        signal_dict[folder_name] = signals
        time_dict[folder_name] = time_line
        fs_dict[folder_name] = fs_signal
    
    return signal_dict, time_dict, fs_dict

# Load all signals
print("Loading physiological signals...")
signal_data, time_data, fs_dict = read_signals(dataset_path)
print(f"Loaded {len(signal_data)} subjects: {sorted(signal_data.keys())}")

---
## Plot Body Signals for All 22 Subjects

In [ ]:
# Signal colors
SIGNAL_COLORS = {
    'BVP': '#e74c3c',
    'HR': '#3498db',
    'ACC': '#e67e22'
}

# Stress phase colors (only stress phases, not rest)
PHASE_COLORS = {
    'Stroop': '#e74c3c',
    'TMCT': '#e67e22',
    'Real Opinion': '#f39c12',
    'Opposite Opinion': '#f1c40f',
    'Subtract': '#d35400'
}

def get_stress_rest_segments(subject_id, tags):
    """Get protocol segments with phase labels."""
    segments = []
    
    if subject_id.startswith('S'):  # V1 protocol
        if len(tags) >= 13:
            segments.append({'start': tags[0], 'end': tags[3], 'label': 'REST', 'phase': 'Baseline'})
            segments.append({'start': tags[3], 'end': tags[4], 'label': 'STRESS', 'phase': 'Stroop'})
            segments.append({'start': tags[4], 'end': tags[5], 'label': 'REST', 'phase': 'First Rest'})
            segments.append({'start': tags[5], 'end': tags[6], 'label': 'STRESS', 'phase': 'TMCT'})
            segments.append({'start': tags[6], 'end': tags[7], 'label': 'REST', 'phase': 'Second Rest'})
            segments.append({'start': tags[7], 'end': tags[8], 'label': 'STRESS', 'phase': 'Real Opinion'})
            segments.append({'start': tags[8], 'end': tags[9], 'label': 'REST', 'phase': 'Transition Rest 1'})
            segments.append({'start': tags[9], 'end': tags[10], 'label': 'STRESS', 'phase': 'Opposite Opinion'})
            segments.append({'start': tags[10], 'end': tags[11], 'label': 'REST', 'phase': 'Transition Rest 2'})
            segments.append({'start': tags[11], 'end': tags[12], 'label': 'STRESS', 'phase': 'Subtract'})
    else:  # V2 protocol
        if len(tags) >= 10:
            segments.append({'start': tags[0], 'end': tags[2], 'label': 'REST', 'phase': 'Baseline'})
            segments.append({'start': tags[2], 'end': tags[3], 'label': 'STRESS', 'phase': 'TMCT'})
            segments.append({'start': tags[3], 'end': tags[4], 'label': 'REST', 'phase': 'First Rest'})
            segments.append({'start': tags[4], 'end': tags[5], 'label': 'STRESS', 'phase': 'Real Opinion'})
            segments.append({'start': tags[5], 'end': tags[6], 'label': 'REST', 'phase': 'Transition Rest'})
            segments.append({'start': tags[6], 'end': tags[7], 'label': 'STRESS', 'phase': 'Opposite Opinion'})
            segments.append({'start': tags[7], 'end': tags[8], 'label': 'REST', 'phase': 'Second Rest'})
            segments.append({'start': tags[8], 'end': tags[9], 'label': 'STRESS', 'phase': 'Subtract'})
    
    return segments

def plot_subject_signals(subject_id, signals, time_dict, tags):
    """
    Plot BVP, HR, ACC signals for one subject with stress phases highlighted.
    """
    protocol = "V1 (Male, with Stroop)" if subject_id.startswith('S') else "V2 (Female, without Stroop)"
    segments = get_stress_rest_segments(subject_id, tags)
    
    fig, axes = plt.subplots(3, 1, figsize=(14, 9), sharex=True)
    fig.suptitle(f'{subject_id} - {protocol}', fontsize=14, fontweight='bold', y=1.01)
    
    signal_names = ['BVP', 'HR', 'ACC']
    signal_labels = [
        'BVP (Blood Volume Pulse)',
        'HR (Heart Rate)',
        'ACC (Movement)'
    ]
    
    for ax, signal_name, signal_label in zip(axes, signal_names, signal_labels):
        # Plot signal
        if signal_name in signals and signal_name in time_dict:
            if signal_name == 'ACC':
                acc_filtered = moving_average(signals[signal_name])
                time_acc = np.linspace(0, len(signals[signal_name])/32, len(acc_filtered))
                ax.plot(time_acc, acc_filtered, color=SIGNAL_COLORS[signal_name], linewidth=0.8)
            else:
                ax.plot(time_dict[signal_name], signals[signal_name], 
                       color=SIGNAL_COLORS[signal_name], linewidth=0.5)
        
        # Highlight STRESS phases only
        for seg in segments:
            if seg['phase'] in PHASE_COLORS:  # Only stress phases
                ax.axvspan(seg['start'], seg['end'], 
                          color=PHASE_COLORS[seg['phase']], alpha=0.3)
        
        ax.set_ylabel(signal_name, fontsize=10, fontweight='bold')
        ax.grid(True, alpha=0.3)
    
    axes[-1].set_xlabel('Time (seconds)', fontsize=11)
    
    # Add legend for signals and phases
    from matplotlib.lines import Line2D
    from matplotlib.patches import Patch
    
    legend_elements = []
    # Signals
    legend_elements.append(Line2D([0], [0], color='none', label='Signals:'))
    for s, l in zip(signal_names, signal_labels):
        legend_elements.append(Line2D([0], [0], color=SIGNAL_COLORS[s], linewidth=2, label=f'  {l}'))
    
    # Stress phases
    legend_elements.append(Patch(facecolor='none', edgecolor='none', label=''))
    legend_elements.append(Patch(facecolor='none', edgecolor='none', label='STRESS Phases:'))
    for phase, color in PHASE_COLORS.items():
        legend_elements.append(Patch(facecolor=color, alpha=0.5, label=f'  {phase}'))
    
    fig.legend(handles=legend_elements, loc='center left', bbox_to_anchor=(1.01, 0.5),
               fontsize=9, title='Legend', title_fontsize=10)
    
    plt.tight_layout()
    plt.subplots_adjust(right=0.82)
    plt.show()

print("Plotting function defined")

### V1 Subjects (Males with Stroop Test)

In [ ]:
for subject_id in v1_subjects:
    if subject_id in signal_data:
        plot_subject_signals(
            subject_id,
            signal_data[subject_id],
            time_data[subject_id],
            signal_data[subject_id]['tags']
        )
    else:
        print(f"WARNING: {subject_id} not found!")

### V2 Subjects (Females without Stroop Test)

In [ ]:
for subject_id in v2_subjects:
    if subject_id in signal_data:
        plot_subject_signals(
            subject_id,
            signal_data[subject_id],
            time_data[subject_id],
            signal_data[subject_id]['tags']
        )
    else:
        print(f"WARNING: {subject_id} not found!")